In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Read UKBB data from file ##

### 1.1 Read all columns with blood count values (reported on the initial date of assessment), age, sex and EID ###

In [23]:
## read eid, sex, year-of-birth, date-of-first-assessment
columns_to_read=['eid','31-0.0','34-0.0','52-0.0', '53-0.0']

## blood counts are recorded in columns 30000-30300
n = 30310                                          # Define the upper limit (exclusive)
numbers = list(range(30000, n, 10))                # Create a list from 0 to n with a step of 10
columns_to_read+=[f'{num}-0.0' for num in numbers] # Create columns-to-read list

# blood_count=pd.read_csv('path/to/file/ukb45304.csv',usecols=columns_to_read)
blood_count=pd.read_csv('ukb45304.csv',usecols=columns_to_read)
blood_count.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502490 entries, 0 to 502489
Data columns (total 36 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   eid        502490 non-null  int64  
 1   31-0.0     502489 non-null  float64
 2   34-0.0     502489 non-null  float64
 3   52-0.0     502489 non-null  float64
 4   53-0.0     502489 non-null  object 
 5   30000-0.0  478153 non-null  float64
 6   30010-0.0  478158 non-null  float64
 7   30020-0.0  478158 non-null  float64
 8   30030-0.0  478158 non-null  float64
 9   30040-0.0  478156 non-null  float64
 10  30050-0.0  478155 non-null  float64
 11  30060-0.0  478151 non-null  float64
 12  30070-0.0  478156 non-null  float64
 13  30080-0.0  478155 non-null  float64
 14  30090-0.0  478151 non-null  float64
 15  30100-0.0  478150 non-null  float64
 16  30110-0.0  478150 non-null  float64
 17  30120-0.0  477269 non-null  float64
 18  30130-0.0  477269 non-null  float64
 19  30140-0.0  477269 non-n

### 1.2 Calculate age at date of assessment ###

In [24]:
blood_count = blood_count.dropna(subset=['53-0.0'])

In [25]:
# Convert the 'date' column to datetime
blood_count['53-0.0'] = pd.to_datetime(blood_count['53-0.0'])

#calculate the age
blood_count['age']= blood_count['53-0.0'].dt.year+(blood_count['53-0.0'].dt.month)/12-(blood_count['34-0.0']+blood_count['52-0.0']/12)

#drop the year of birth, date of assessment and year column
columns_to_drop=['34-0.0','52-0.0','53-0.0']
blood_count=blood_count.drop(columns=columns_to_drop)

blood_count.head()

,eid,31-0.0,30000-0.0,30010-0.0,30020-0.0,30030-0.0,30040-0.0,30050-0.0,30060-0.0,30070-0.0,...,30220-0.0,30230-0.0,30240-0.0,30250-0.0,30260-0.0,30270-0.0,30280-0.0,30290-0.0,30300-0.0,age
0,1000010,1.0,11.60,5.140,16.00,46.20,89.90,31.20,34.7,13.60,...,0.60,0.0,2.350,0.121,100.8,78.60,0.370,0.860,0.044,60.166667
1,1000028,0.0,8.30,3.890,12.50,37.50,96.40,32.20,33.4,13.10,...,0.50,0.0,0.620,0.024,108.2,86.30,0.320,0.200,0.008,48.833333
2,1000034,0.0,9.30,4.320,12.60,38.30,88.70,29.20,32.9,13.80,...,0.50,0.0,1.850,0.080,104.4,84.50,0.330,0.620,0.027,69.750000
3,1000045,0.0,5.09,4.395,13.55,41.06,93.43,30.83,33.0,12.25,...,0.55,0.0,1.119,0.049,109.8,83.96,0.253,0.284,0.012,47.583333
4,1000052,1.0,8.40,4.130,14.00,41.00,99.20,34.00,34.2,13.00,...,0.20,0.0,1.660,0.069,113.5,86.20,0.270,0.450,0.018,69.333333


### 1.3 Save the CBC values, age and sex ###

In [26]:
## save in a separate file: blood_count.csv
blood_count.to_csv('blood_count.csv',index=False,float_format='%.3f')

## 2. Exclude patients with prevalent hematological malignancies ## 

### 2.1. Read only the cancer diagnosis columns and eliminate ones with prevalent hematological malignancies: date of diagnosis on or before sample collection date ###

In [3]:
# Step 1: read column names only
df_temp=pd.read_csv('ukb45304.csv',nrows=0)
df_temp.shape

cols1=['eid','53-0.0']

# Step 2: Get columns for cancer code & year of diagnosis
# 20001,20006 Cancer codes and dates- self-reported
# 41270, 41280 ICD codes and corresponding dates

cols_to_read = cols1+df_temp.filter(regex='^20001|^20006').columns.to_list()+df_temp.filter(regex='^41270|^41280').columns.to_list()
#print(cols_to_read)

# Step 3: Read only the required columns
df = pd.read_csv('ukb45304.csv', usecols=cols_to_read)
df.shape

/var/folders/01/qxbg7rcx0ds_3w3r_82z10zh0000gn/T/ipykernel_22732/1997610971.py:15: DtypeWarning: Columns (4348,4349,4350,4351,4352,4353,4354,4355,4356,4357,4358,4359,4360,4361,4362,4363,4364,4365,4366,4367,4368,4369,4370,4371,4372,4373,4374,4375,4376,4377,4378,4379,4380,4381,4382,4383,4384,4385,4386,4387,4388,4389,4390,4391,4392,4393,4394,4395,4396,4397,4398,4399,4400,4401,4402,4403,4404,4405,4406,4407,4408,4409,4410,4411,4412,4413,4414,4415,4416,4417,4418,4419,4420,4421,4422,4423,4424,4425,4426,4427,4428,4429,4430,4431,4432,4433,4434,4435,4436,4437,4438,4439,4440,4441,4442,4443,4444,4445,4446,4447,4448,4449,4450,4451,4452,4453,4454,4455,4456,4457,4458,4459,4460,4461,4462,4463,4464,4465,4466,4467,4468,4469,4470,4471,4472,4473,4474,4475,4476,4477,4478,4479,4480,4481,4482,4483,4484,4485,4486,4487,4488,4489,4490,4491,4492,4493,4494,4495,4496,4497,4498,4499,4500,4501,4502,4503,4504,4505,4506,4507,4508,4509,4510,4511,4512,4513,4514,4515,4516,4517,4518,4519,4520,4521,4522,4523,4524,4608,4609

(502490, 476)

### 2.2 Pre-process columns ###

In [4]:
# Convert the 'date' column to datetime
df['53-0.0'] = pd.to_datetime(df['53-0.0'])
# extract the year
df['year_of_assessment']=df['53-0.0'].dt.year+(df['53-0.0'].dt.month/12)
# drop the date column
df=df.drop(columns='53-0.0')

In [5]:
## for the self-reported cancer columns 20001, codes for hematological malignancies
hematopoietic_malignancy=[1047,1048,1050,1051,1052,1053,1055,1056,1058]

## ICD codes for hematological malognancies
MDS=['D460','D461','D462','D463','D464','D465','D466','D467','D469']
AML=['C920','C923','C924','C926','C930','C937','C939','C940','C942','C943','C944','C950']
MPN=['D45','D473','C946','D474']
CML=['C922','C931','C932','C941','C951','C952','D475','C957','C947','C937','C927','C959','C929']
Ly_lu=['C910','C911','C912','C913','C914','C915','C917','C918','C919']
Lymphoma=['C810','C811','C812','C813','C814','C817','C819','C820','C821','C822','C823','C824','C825','C826','C827','C829','C830', 
         'C831','C833','C834','C835','C836','C837','C839','C840','C841','C842','C843','C844','C845','C846','C847','C848','C849','C860',
          'C861','C862','C863','C864','C865','C866']
diagnoses_HM=MDS+AML+MPN+CML+Ly_lu+Lymphoma
#print(diagnoses_HM)

In [6]:
## Check the dtypes of cancer columns and dates 
first_valid_icd = df['41280-0.0'].dropna().iloc[0]
print(first_valid_icd)
first_valid_cancer = df['20006-0.0'].dropna().iloc[0]
print(first_valid_cancer)

2019-05-22
2002.5


*The cancer diagnosis dates are already in float format, the ICD diagonosis dates are in datetime format*

In [7]:
# Convert the date column for ICD diagnosis

ICD_diagnosis_dates=df_temp.filter(regex='^41280').columns.to_list()

for i in ICD_diagnosis_dates:
    df[i] = pd.to_datetime(df[i])
    df[i]=df[i].dt.year+(df[i].dt.month/12)

In [60]:
# Check length of list of columns

ICD_cancer_codes=df_temp.filter(regex='^41270').columns.to_list()
SD_cancer_codes=df_temp.filter(regex='^20001').columns.to_list()

print('length of cancer columns:',len(SD_cancer_codes))
print('length of ICD columns:',len(ICD_cancer_codes))

# Check
'''
for i in SD_cancer_codes:
    #get column index
    col_index=df.columns.get_loc(i)
    # Pick the column 24/213 places after
    new_col = df.columns[col_index + 24]
    print(i,new_col)
'''    

length of cancer columns: 24
length of ICD columns: 213


'\nfor i in SD_cancer_codes:\n    #get column index\n    col_index=df.columns.get_loc(i)\n    # Pick the column 24/213 places after\n    new_col = df.columns[col_index + 24]\n    print(i,new_col)\n'

### 2.3 Remove rows where MN diagnosis date is before date of assessment ###

In [62]:
df_updated = df.copy()
original_count = len(df_updated)

# ---------- 1) Self‑reported codes ----------
for code in SD_cancer_codes:
    # Get column index 
    col_idx  = df_updated.columns.get_loc(code)
    date_col = df_updated.columns[col_idx + 24]

    mask = (
        df_updated[code].isin(hematopoietic_malignancy) &
        (df_updated[date_col] <= df_updated['year_of_assessment'])
    )

    df_updated = df_updated.loc[~mask]

after_removing_SDHM = len(df_updated)
print("cases of prevalent malignancies in self‑reported cases:",original_count - after_removing_SDHM)

# ---------- 2) ICD codes ----------
for code in ICD_cancer_codes:
    col_idx  = df_updated.columns.get_loc(code)
    date_col = df_updated.columns[col_idx + 213]

    mask = (
        df_updated[code].isin(diagnoses_HM) &
        (df_updated[date_col] <= df_updated['year_of_assessment'])
    )

    df_updated = df_updated.loc[~mask]

after_removing_ICDHM = len(df_updated)
print("cases of prevalent malignancies in ICD diagnosis:", after_removing_SDHM - after_removing_ICDHM)

cases of prevalent malignancies in self‑reported cases: 2468
cases of prevalent malignancies in ICD diagnosis: 500


In [63]:
print('Remaining cohort count after removing patients with prevalent HM:', after_removing_ICDHM)

Remaining cohort count after removing patients with prevalent HM: 499522


### 2.4 Create rows indicating incident hematologic malignancies, dates and types ###

In [64]:
df_updated.head()

,eid,20001-0.0,20001-0.1,20001-0.2,20001-0.3,20001-0.4,20001-0.5,20001-1.0,20001-1.1,20001-1.2,...,41280-0.204,41280-0.205,41280-0.206,41280-0.207,41280-0.208,41280-0.209,41280-0.210,41280-0.211,41280-0.212,year_of_assessment
0,1000010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007.916667
1,1000028,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008.583333
2,1000034,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008.666667
3,1000045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2009.333333
4,1000052,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007.833333


In [73]:
df_ar=df_updated.to_numpy()

cancer_code_idx = [df_updated.columns.get_loc(c) for c in SD_cancer_codes]
ICD_code_idx=[df_updated.columns.get_loc(c) for c in ICD_cancer_codes]
cancer_date_idx = [df_updated.columns.get_loc(c) + 24 for c in SD_cancer_codes]
ICD_date_idx = [df_updated.columns.get_loc(c) + 213 for c in ICD_cancer_codes]

# ----- preparation ----------------------------------------------------------
N = len(df_updated)                               # number of rows
sentinel_date = np.nan                            # for “no hit yet”

earliest_date = np.full(N, np.nan)                # float array (NaN = no hit)
earliest_code = np.full(N, None, dtype=object)    # object array for strings / ints
earliest_is_sd = np.zeros(N, dtype=bool)          # track whether first hit is SD

# ----- pass 1 : scan SD columns ---------------------------------------------
for code_idx, date_idx in zip(cancer_code_idx, cancer_date_idx):
    mask = (np.isin(df_ar[:, code_idx], hematopoietic_malignancy) & (df_ar[:, date_idx] > df_ar[:, assessment_idx]))
    rows = np.where(mask)[0]
    for r in rows:
        d = df_ar[r, date_idx]
        if np.isnan(earliest_date[r]) or d < earliest_date[r]:
            earliest_date[r] = d
            earliest_code[r] = df_ar[r, code_idx]
            earliest_is_sd[r] = True              # remember SD came first

print('Self reported codes checked')

# ----- pass 2 : scan ICD columns --------------------------------------------
for code_idx, date_idx in zip(ICD_code_idx, ICD_date_idx):
    mask = (np.isin(df_ar[:, code_idx], diagnoses_HM) & (df_ar[:, date_idx] > df_ar[:, assessment_idx]))
    rows = np.where(mask)[0]
    #print(len(rows))
    for r in rows:
        d = df_ar[r, date_idx]
        #   overwrite only if:
        #   • we already have an SD hit
        #   • this ICD date is earlier than any SD date stored
        
        if earliest_is_sd[r]:
            earliest_date[r] = d
            earliest_code[r] = df_ar[r, code_idx]
            earliest_is_sd[r] = False             # now first hit is ICD

        # if we have *no* hit yet, just take the ICD one
        elif np.isnan(earliest_date[r]) or d < earliest_date[r]:
            earliest_date[r] = d
            earliest_code[r] = df_ar[r, code_idx]
            earliest_is_sd[r] = False

print('ICD codes checked')
# ----- write back to DataFrame ----------------------------------------------
df_updated["year_HM"] = earliest_date
df_updated["HM_type"] = earliest_code

Self reported codes checked
1518
1300
933
689
567
350
260
156
100
78
39
36
28
17
10
10
8
4
4
2
2
1
1
1
0
0
2
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
ICD codes checked


In [74]:
df_updated.head()

,eid,20001-0.0,20001-0.1,20001-0.2,20001-0.3,20001-0.4,20001-0.5,20001-1.0,20001-1.1,20001-1.2,...,41280-0.206,41280-0.207,41280-0.208,41280-0.209,41280-0.210,41280-0.211,41280-0.212,year_of_assessment,year_HM,HM_type
0,1000010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007.916667,NaN,None
1,1000028,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008.583333,NaN,None
2,1000034,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008.666667,2014.75,C911
3,1000045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2009.333333,NaN,None
4,1000052,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007.833333,NaN,None


### 2.5 Drop all codes and dates ###

In [75]:
cols_to_drop = df_temp.filter(regex='^20001|^20006|^41270|^41280').columns.to_list()
df_HM_removed=df_updated.drop(columns=cols_to_drop)
print(df_HM_removed.info())

<class 'pandas.core.frame.DataFrame'>
Index: 499522 entries, 0 to 502489
Data columns (total 4 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   eid                 499522 non-null  int64  
 1   year_of_assessment  499521 non-null  float64
 2   year_HM             4660 non-null    float64
 3   HM_type             4660 non-null    object 
dtypes: float64(2), int64(1), object(1)
memory usage: 19.1+ MB
None


In [76]:
df_HM_removed.to_csv('HM_removed_id.csv', index=False)

## 3. Gene Data ##

In [19]:
gene_data=pd.read_csv('ukbvariantcalls.csv')
gene_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 470960 entries, 0 to 470959
Data columns (total 16 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Broad_ID  470960 non-null  int64  
 1   gene1     29835 non-null   object 
 2   gene2     2066 non-null    object 
 3   gene3     209 non-null     object 
 4   gene4     0 non-null       float64
 5   gene5     0 non-null       float64
 6   gene6     0 non-null       float64
 7   VAF1      29835 non-null   float64
 8   VAF2      2066 non-null    float64
 9   VAF3      209 non-null     float64
 10  VAF4      0 non-null       float64
 11  VAF5      0 non-null       float64
 12  VAF6      0 non-null       float64
 13  31-0.0    470960 non-null  int64  
 14  MNeo      470960 non-null  int64  
 15  NonMTime  69851 non-null   float64
dtypes: float64(10), int64(3), object(3)
memory usage: 57.5+ MB


In [20]:
columns_to_remove=['gene4','gene5','gene6','VAF4','VAF5','VAF6','MNeo','NonMTime']
gene_data2=gene_data.drop(columns=columns_to_remove)
len(gene_data2)

470960

### 3.1 Molecular CH score calculation ###

In [21]:
CH_gene_list_weeks = [ 
"ASXL1", "ASXL2", "BCOR", "BCORL1", "BRAF", "BRCC3", "CBL", "CEBPA", "CREBBP", "CTCF",
"CUX1", "DNMT3A", "EED", "EP300", "ETNK1", "ETV6", "EZH2", "FLT3", "GATA1", "GATA2", "GATA3", 
"GNA13", "GNAS", "GNB1", "IDH1", "IDH2", "IKZF1", "IKZF2", "IKZF3", "JAK1", "JAK2", "JAK3",
"KDM6A", "KIT", "KRAS", "LUC7L2", "MPL", "NF1", "NPM1", "NRAS", "NDF1", "PDSS6", "PHF6", "PHIP",
"PPM1D", "PRPF8", "PTEN", "PTPN11", "RAD21", "RUNX1", "SETBP1", "SETD2", "SETDB1", "SF3B1", "SRSF2", 
"SMC1A", "SMC3", "SRCAP", "STAG1", "STAG2", "SUZ12", "TET2", "TP53", "U2AF1*", "U2AF2", "WT1", "YPL1H1",
"ZBTB33", "ZNF318", "ZRSR2"
]

CH_highrisk=["JAK2","SRSF2","SF3B1","ZRSR2","RUNX1","IDH2","IDH1","FLT3","TP53"]
print(len(CH_gene_list_weeks),len(CH_highrisk))

70 9


In [22]:
gene_ar=gene_data2.to_numpy()
rows,columns=gene_ar.shape
print(rows,columns)

#initialize the outcome column
CH_score=np.zeros((rows,))
CH_list=[]
count=0

for i in range(rows):
    # Check for presence of CH
    candidates = [
        (gene_ar[i, j], gene_ar[i, j + 3])
        for j in range(1, 4)
        if gene_ar[i, j] in CH_gene_list_weeks and gene_ar[i, j + 3] >= 0.02
    ]
    if candidates:
        count+=1
        # Check VAF >20%
        if any(r[1]>=0.2 for r in candidates):
            CH_score[i]+=2
        else:    
            CH_score[i]+=1  
                
        #---------------------------------------------- 
        
        # Check multiple mutations
        if pd.Series(gene_ar[i, 1:4]).notna().sum() == 1:
            CH_score[i] += 1
        elif pd.Series(gene_ar[i, 1:4]).notna().sum() > 1:
            CH_score[i] += 2
            
        #---------------------------------------------- 
        
        # Check single DNMT3A
        if (np.sum(gene_ar[i,1:4] == 'DNMT3A')==1)&(pd.Series(gene_ar[i, 1:4]).notna().sum() == 1):
            CH_score[i]+=0.5
        else:
            CH_score[i]+=1
    
        #-----------------------------------------------
    
        # Check high risk mutation
        if any(value in CH_highrisk for value in gene_ar[i,1:4]):
            CH_score[i]+=2.5
        else:    
            CH_score[i]+=1
    CH_list.append([gene_ar[i, 0],gene_ar[i, 1],gene_ar[i, 2],gene_ar[i, 3], gene_ar[i, 4],gene_ar[i, 5],gene_ar[i, 6], gene_ar[i, 7], CH_score[i]])

print('population with CH:', count)

CH_data=pd.DataFrame(CH_list,columns=['Broad_ID','gene1','gene2','gene3','VAF1','VAF2','VAF3','31-0.0','CH_score'])
CH_data.head()

470960 8
population with CH: 29526


,Broad_ID,gene1,gene2,gene3,VAF1,VAF2,VAF3,31-0.0,CH_score
0,3463778,NaN,NaN,NaN,NaN,NaN,NaN,1,0.0
1,2821197,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0
2,1514525,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0
3,2491905,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0
4,3250377,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0


## 4. Link Files to create the final dataset ##

In [77]:
linker=pd.read_csv('linker.csv')
linker=linker.drop(columns=['Unnamed: 0'])

# Merge gene_data and linker on Broad_id
merged1 = pd.merge(CH_data, linker, on='Broad_ID')

#merged1.head()

# Merge merged1 with blood_counts on eid
merged2 = pd.merge(merged1, blood_count, on=['eid','31-0.0'])
merged2.head()

print('Number of patients with CH data:', len(merged2))

# Only keep IDs without prevalent HM
merged_df = pd.merge(merged2, df_HM_removed, on=['eid'],how='inner')

print('Final number of patients in the cohort after exclusion of prevalent HM and with CH information:', len(merged_df))

merged_df.head()

Number of patients with CH data: 470960
Final number of patients in the cohort after exclusion of prevalent HM and with CH information: 468695


,Broad_ID,gene1,gene2,gene3,VAF1,VAF2,VAF3,31-0.0,CH_score,eid,...,30250-0.0,30260-0.0,30270-0.0,30280-0.0,30290-0.0,30300-0.0,age,year_of_assessment,year_HM,HM_type
0,3463778,NaN,NaN,NaN,NaN,NaN,NaN,1,0.0,4726860.0,...,0.064,111.23,82.09,0.211,0.284,0.014,42.000000,2009.666667,NaN,None
1,2821197,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,1633945.0,...,0.040,104.46,81.47,0.214,0.180,0.009,47.000000,2009.666667,NaN,None
2,1514525,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,4596360.0,...,0.020,106.56,86.68,0.228,0.103,0.005,52.416667,2009.666667,NaN,None
3,2491905,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,3143018.0,...,0.084,111.75,83.24,0.360,0.658,0.030,65.750000,2009.666667,NaN,None
4,3250377,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,2254568.0,...,0.078,107.37,82.90,0.271,0.452,0.021,57.000000,2009.666667,NaN,None


### 4.1 Change column names ###

In [78]:
## change column names 
column_names=['Broad_ID','gene1','gene2','gene3','VAF1','VAF2','VAF3','sex','CH_score','eid','WBC','RBC','Hbconc',
              'haematocrit','MCV','MCHb','MCHbconc','RDW','platelet','plateletcrit','MPV','PDW','lymphocyte',
              'monocyte','neutrophil','eosinophil','basophil','nucRBC','lymph%','mono%','neutro%','eosin%',
              'bas%','nucRBC%','retic%','reticulocyte','MRV','MspCV','immret','hiscatret%','hiscatret','age',
              'year_of_assessment','year_HM','HM_type']
merged_df.columns = column_names
merged_df.head()

,Broad_ID,gene1,gene2,gene3,VAF1,VAF2,VAF3,sex,CH_score,eid,...,reticulocyte,MRV,MspCV,immret,hiscatret%,hiscatret,age,year_of_assessment,year_HM,HM_type
0,3463778,NaN,NaN,NaN,NaN,NaN,NaN,1,0.0,4726860.0,...,0.064,111.23,82.09,0.211,0.284,0.014,42.000000,2009.666667,NaN,None
1,2821197,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,1633945.0,...,0.040,104.46,81.47,0.214,0.180,0.009,47.000000,2009.666667,NaN,None
2,1514525,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,4596360.0,...,0.020,106.56,86.68,0.228,0.103,0.005,52.416667,2009.666667,NaN,None
3,2491905,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,3143018.0,...,0.084,111.75,83.24,0.360,0.658,0.030,65.750000,2009.666667,NaN,None
4,3250377,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,2254568.0,...,0.078,107.37,82.90,0.271,0.452,0.021,57.000000,2009.666667,NaN,None


## 5. Calculate total CH risk score and MN status ##

### 5.1 CHRS ###

In [79]:
df_ar=merged_df.to_numpy()
rows,columns=df_ar.shape
print(rows,columns)

#initialize the outcome column
CH_score=np.zeros((rows,))
CH_list=[]

RDW_index=merged_df.columns.get_loc('RDW')
MCV_index=merged_df.columns.get_loc('MCV')
age_index=merged_df.columns.get_loc('age')
chscore_index=merged_df.columns.get_loc('CH_score')

# CHIP/CCUS condition

anemia= np.where(((merged_df['sex']==0)&(merged_df['Hbconc']<12))|((merged_df['sex']==1)&(merged_df['Hbconc']<13)), True, False)
neutropenia= np.where(merged_df['neutrophil']<1.8, True, False)                                            
thrombocytopenia= np.where(merged_df['platelet']<150, True, False)

for i in range(rows):
    
    if (df_ar[i,chscore_index]>0):

        # Check RDW 
        if df_ar[i,RDW_index]<15:
            CH_score[i]+=1
        else:
            CH_score[i]+=2.5
    
        #-------------------------------------------
        #Check MVC
        if df_ar[i,MCV_index]<100:
            CH_score[i]+=1
        else:
            CH_score[i]+=2.5
    
        #-------------------------------------------
        #Check age
        if df_ar[i,age_index]<65:
            CH_score[i]+=1
        else:
            CH_score[i]+=1.5
        #-------------------------------------------

CH_score[(CH_score>0)&(anemia|neutropenia|thrombocytopenia)]+=1.5
CH_score[(CH_score>0)&(~(anemia|neutropenia|thrombocytopenia))]+=1

merged_df['CHRS']=merged_df['CH_score']+CH_score

468695 45


### 5.2 MN status assignment ###

In [85]:
# Assign condition for MN 
MDS=['D460','D461','D462','D463','D464','D465','D466','D467','D469']
AML=['C920','C923','C924','C926','C930','C937','C939','C940','C942','C943','C944','C950']
MPN=['D45','D473','C946','D474']
CML=['C922','C931','C932','C941','C951','C952','D475','C957','C947','C937','C927','C959','C929']

MN=MDS+AML+MPN+CML


df_CH=merged_df.copy()
df_CH.loc[:,'MN']=np.where((df_CH['HM_type'].isin(MN))&(df_CH['year_HM'].notnull()),1,0)
df_CH.head()

,Broad_ID,gene1,gene2,gene3,VAF1,VAF2,VAF3,sex,CH_score,eid,...,MspCV,immret,hiscatret%,hiscatret,age,year_of_assessment,year_HM,HM_type,CHRS,MN
0,3463778,NaN,NaN,NaN,NaN,NaN,NaN,1,0.0,4726860.0,...,82.09,0.211,0.284,0.014,42.000000,2009.666667,NaN,None,0.0,0
1,2821197,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,1633945.0,...,81.47,0.214,0.180,0.009,47.000000,2009.666667,NaN,None,0.0,0
2,1514525,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,4596360.0,...,86.68,0.228,0.103,0.005,52.416667,2009.666667,NaN,None,0.0,0
3,2491905,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,3143018.0,...,83.24,0.360,0.658,0.030,65.750000,2009.666667,NaN,None,0.0,0
4,3250377,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,2254568.0,...,82.90,0.271,0.452,0.021,57.000000,2009.666667,NaN,None,0.0,0


## 6. Remove Null Values ##

In [86]:
df_CH.isnull().sum()

Broad_ID                   0
gene1                 439137
gene2                 466675
gene3                 468490
VAF1                  439137
VAF2                  466675
VAF3                  468490
sex                        0
CH_score                   0
eid                        0
WBC                        0
RBC                        1
Hbconc                     0
haematocrit                1
MCV                        3
MCHb                       3
MCHbconc                   7
RDW                        3
platelet                   0
plateletcrit               4
MPV                        5
PDW                        5
lymphocyte                 0
monocyte                   0
neutrophil                 0
eosinophil                 0
basophil                   0
nucRBC                     9
lymph%                     0
mono%                      0
neutro%                    0
eosin%                     0
bas%                       0
nucRBC%                   13
retic%        

In [89]:
df_CH_nullremoved=df_CH.dropna(subset=['MCHbconc','MPV','nucRBC','reticulocyte','MRV'])

df_CH_nullremoved.isnull().sum()

Broad_ID                   0
gene1                 432499
gene2                 459631
gene3                 461416
VAF1                  432499
VAF2                  459631
VAF3                  461416
sex                        0
CH_score                   0
eid                        0
WBC                        0
RBC                        0
Hbconc                     0
haematocrit                0
MCV                        0
MCHb                       0
MCHbconc                   0
RDW                        0
platelet                   0
plateletcrit               0
MPV                        0
PDW                        0
lymphocyte                 0
monocyte                   0
neutrophil                 0
eosinophil                 0
basophil                   0
nucRBC                     0
lymph%                     0
mono%                      0
neutro%                    0
eosin%                     0
bas%                       0
nucRBC%                    4
retic%        

## 7. Drop the blood count percentage values ##

In [90]:
columns_to_drop=['lymph%','mono%','neutro%','eosin%','bas%','nucRBC%','retic%','hiscatret%']
df_CH_final=df_CH_nullremoved.drop(columns=columns_to_drop)

In [91]:
df_CH_final.head()

,Broad_ID,gene1,gene2,gene3,VAF1,VAF2,VAF3,sex,CH_score,eid,...,MRV,MspCV,immret,hiscatret,age,year_of_assessment,year_HM,HM_type,CHRS,MN
0,3463778,NaN,NaN,NaN,NaN,NaN,NaN,1,0.0,4726860.0,...,111.23,82.09,0.211,0.014,42.000000,2009.666667,NaN,None,0.0,0
1,2821197,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,1633945.0,...,104.46,81.47,0.214,0.009,47.000000,2009.666667,NaN,None,0.0,0
2,1514525,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,4596360.0,...,106.56,86.68,0.228,0.005,52.416667,2009.666667,NaN,None,0.0,0
3,2491905,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,3143018.0,...,111.75,83.24,0.360,0.030,65.750000,2009.666667,NaN,None,0.0,0
4,3250377,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,2254568.0,...,107.37,82.90,0.271,0.021,57.000000,2009.666667,NaN,None,0.0,0


In [92]:
df_CH_final.to_csv('UKBB_preprocessed.csv',index=False)